# Animation

**Part I · Visualization** — Tutorial 11

Animate geometric constructions. You will learn to:

- Drive scripted animations with `Visualizer.animate()` (update entities in
  place via the `content` aspect).
- Use `animate(auto_clear=True)` for per-frame `add()` calls.
- Build trails and polylines with `PointPath` and gradient utilities.
- Tween with `animate_to` and choreograph with the scene-aware `Timeline`.


## Setup


In [ ]:
import math

from pytanga.geometry import Point
from pytanga.viz import (
    PointPath, PointPathStyle, PointStyle, Visualizer, gradient_colors,
)


## 1. `animate()` — the frame loop

`animate()` is a generator that yields once per frame (the elapsed wall-clock
time) and paces to a target `fps`. It **does not open the viewer** — call
`show()` first (or use `with viz:`). Update entities in place so only changed
entities are pushed.


In [ ]:
viz = Visualizer(title="Animation — orbit", add_default_axes=False, add_default_grid=False)
viz.show()

p = viz.new(Point(3, 0, 0), color="#ff4444", label="orbit")

angle = 0.0
for dt in viz.animate(fps=30):
    angle += 3.0 * dt
    p.entity = Point(3 * math.cos(angle), 3 * math.sin(angle), 0)   # "content" aspect
    viz.flush()
    if angle > 2 * math.pi:
        break

viz.stop_server()
viz.display_snapshot()


## 2. `animate(auto_clear=True)`

For per-frame `add()` calls that should **not** accumulate, pass
`auto_clear=True` — each frame flushes then removes objects added after the loop
began (anything added *before* the loop persists).


In [ ]:
viz = Visualizer(title="Animation — auto_clear", add_default_axes=False, add_default_grid=False)
viz.show()
viz.new(Point(0, 0, 0), color="#ffffff", label="centre")   # persists

angle = 0.0
for dt in viz.animate(fps=30, auto_clear=True):
    angle += 3.0 * dt
    viz.new(Point(3 * math.cos(angle), 3 * math.sin(angle), 0), color="#ff4444")
    viz.flush()
    if angle > 2 * math.pi:
        break

viz.stop_server()
viz.display_snapshot()


## 3. `PointPath` — trails and polylines

`PointPath` renders an ordered list of points as connected line segments. It
supports FIFO capping (`max_points`), per-point colors, and a `PointPathStyle`
(`color`, `opacity`, screen-space `line_thickness`).


In [ ]:
viz = Visualizer(title="Animation — PointPath", add_default_axes=False, add_default_grid=False)

path = PointPath()
path.add((0, 0, 0), color="#ff0000")
path.add((1, 2, 0), color="#00ff00")
path.add(Point(3, 1, 0), color="#0000ff")
viz.add(path, style=PointPathStyle(line_thickness=3))

viz.display_snapshot()


## 4. Gradient trails and FIFO capping

`gradient_colors(start, end, steps)` interpolates RGB; with `pop_colors=False`,
`default_colors` stay anchored to position slots so the tail fades while the
head stays bright.


In [ ]:
TRAIL_LENGTH = 40
gradient = gradient_colors("#440000", "#ffaa00", TRAIL_LENGTH)

viz = Visualizer(title="Animation — gradient trail", add_default_axes=False, add_default_grid=False)

trail = PointPath(max_points=TRAIL_LENGTH, pop_colors=False, default_colors=gradient)
for _ in range(TRAIL_LENGTH):          # pre-fill so it draws immediately
    trail.add((0, 0, 0))

point = viz.new(Point(3, 0, 0), color="#ffaa00", style=PointStyle(size=0.12))
trail_ref = viz.new(trail, style=PointPathStyle(line_thickness=2))

angle = 0.0
for _ in viz.animate(fps=30):
    angle += 0.1
    x, y = 3 * math.cos(angle), 3 * math.sin(angle)
    point.entity = Point(x, y, 0)
    trail.add((x, y, 0))
    trail_ref.entity = trail
    viz.flush()
    if angle > 2 * math.pi:
        break

viz.stop_server()
viz.display_snapshot()


## 5. Keyframe tweening — `animate_to`

Smooth browser-driven transitions without a Python loop. `animate_to` tweens
position / rotation / opacity / scale with a duration and easing.


In [ ]:
viz = Visualizer(title="Animation — tween", add_default_axes=False, add_default_grid=False)
pid = viz.add(Point(0, 0, 0), color="#ff4444", label="P")

viz.animate_to(pid, position=(5, 0, 0), duration=1.5, easing="ease-out")

viz.flush()
viz.export_snapshot("_output/11_tween.html", overwrite=True)
print("tween scheduled (runs in the live viewer)")


## 6. `Timeline` — choreographed animations

`viz.timeline()` returns a fluent `Timeline`: `wait(seconds)`, `animate_to(
..., parallel=True)`, and `play()`. Timelines created through a
`VizSceneHandle` are automatically scoped to that scene.


In [ ]:
viz = Visualizer(title="Animation — timeline", add_default_axes=False, add_default_grid=False)

p1 = viz.add(Point(0, 0, 0), color="#ff4444", opacity=0.0, label="$P_1$")
p2 = viz.add(Point(5, 0, 0), color="#44ff44", opacity=0.0, label="$P_2$")

viz.timeline() \
    .wait(0.5) \
    .animate_to(p1, opacity=1.0, duration=0.3) \
    .animate_to(p2, opacity=1.0, duration=0.3, parallel=True) \
    .animate_to(p1, position=(3, 2, 0), duration=1.5, easing="ease-out") \
    .animate_to(p2, position=(0, 3, 0), duration=2.0, parallel=True) \
    .play()

viz.flush()
viz.export_snapshot("_output/11_timeline.html", overwrite=True)
print("timeline played (runs in the live viewer)")


## 7. Live plotting trails

Drive live plotting trails with `CoordinateSystem.add_plot()` /
`update_plots()` — see [Tutorial 08](../08_coordinate_system/).


## Visual Examples

An animated orbit recorded with `start_animation_recording()` and exported as a
self-contained HTML file with embedded JS playback controls.


In [ ]:
from pytanga.viz import AnimStyle

viz = Visualizer(title="Animation — animated export", add_default_axes=False, add_default_grid=False)
viz.show()

point = viz.new(Point(3, 0, 0), color="#ff4444", label="orbit")
viz.flush()

recording = viz.start_animation_recording()
for frame in range(90):
    angle = frame * 0.07
    point.entity = Point(3 * math.cos(angle), 3 * math.sin(angle), 0)
    viz.flush()
    recording.capture_frame()
    viz.sleep_ms(33)

viz.export_snapshot(
    "_output/11_orbit.html",
    overwrite=True,
    animation=recording,
    anim_style=AnimStyle(fps=30, loop=True, compress=True),
)
viz.stop_server()
print("animated orbit exported")


## Summary

| Task | API |
|---|---|
| Frame loop | `for dt in viz.animate(fps=...): ...` |
| Update in place | `ref.entity = new_entity` (content aspect) |
| Per-frame add | `viz.animate(auto_clear=True)` |
| Polylines / trails | `PointPath` + `PointPathStyle` |
| Gradient | `gradient_colors` / `multi_gradient_colors` |
| Tween | `viz.animate_to(id, position=..., duration=...)` |
| Choreography | `viz.timeline().wait(...).animate_to(...).play()` |
| Live plots | `CoordinateSystem.add_plot` / `update_plots` |

**Next:** [12 — Split Views](../12_split_views/).
